# BC–WA PM2.5 forecasting benchmark on Colab

Trains AirLapseV2 and baseline models on the BC + Washington hourly PM2.5
dataset that ships in this repo (`data/`), and scores 24-hour forecasts.

**Runtime → Change runtime type → GPU** before running. All metric files are
saved to Google Drive (`METRICS_DIR`), so a disconnected runtime loses no results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# metrics (benchmark.csv, per-run metric files, logs) -> Drive; the large
# prediction arrays and checkpoints stay on the runtime's local disk
METRICS_DIR = '/content/drive/MyDrive/bcwa_pm25_results'
LOCAL_RESULTS = '/content/results'

%cd /content
!rm -rf BC-WA-PM2.5-Forecast && git clone -q https://github.com/Grace-VN/BC-WA-PM2.5-Forecast.git
%cd /content/BC-WA-PM2.5-Forecast
!pip install -q -r requirements.txt
import os, torch
os.environ['METRICS_DIR'] = METRICS_DIR
os.environ['RESULTS_DIR'] = LOCAL_RESULTS
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## Quick check (a few minutes)

One epoch of GRU, just to confirm everything runs. Its scores are meaningless.

In [ ]:
!METRICS_DIR="{METRICS_DIR}/smoke_test" python benchmark.py --models GRU --epochs 1 --repeats 1

## Benchmark

Each model trains `--repeats` times; `benchmark.csv` in `METRICS_DIR` collects
the scores, `runs/<model>/` keeps train.py's own metric file per run, and
`logs/` the full training output - all on Drive. The cell is quiet until each
model finishes. If the runtime disconnects, re-run listing only the
models that hadn't finished - finished ones are kept.

Start with `--repeats 1` to see timings, then use 3-5 for the paper.

The 11 models: AirLapseV2 (proposed); LSTM, Transformer, Informer; PM25_GNN,
AirFormer, AirPhyNet, AirDualODE, AirDDE; and two recent PM2.5 models
re-implemented from their papers, TCN_DIR (Seo et al. 2026) and STMamba
(Zhang et al. 2025). Expect several hours on a T4 for one repeat of all 11 -
if the free runtime stops, re-run the cell with only the unfinished models.


In [ ]:
# Fast models first; the slow ones (AirDDE, AirFormer, TCN_DIR) last, so a
# disconnect costs the least. Re-run with only the unfinished names if needed.
MODELS = 'LSTM PM25_GNN AirLapseV2 AirPhyNet Transformer AirDualODE STMamba Informer TCN_DIR AirFormer AirDDE'
!python benchmark.py --models {MODELS} --epochs 50 --repeats 1 --early_stop 10


In [ ]:
import pandas as pd
df = pd.read_csv(f'{METRICS_DIR}/benchmark.csv')
df = df[df.status == 'ok']
df.groupby('model')[['RMSE', 'MAE', 'MAPE', 'RMSE_gt35.5', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']].mean().round(3).sort_values('RMSE')